# IE-OPT-11 · 单机总延误（位置表示）

**场景**：青禾电器只有一台检测台。四台返修样机必须一台接一台检测，不能并行，也不能中途换下一台。每台有检测工时和承诺完成时刻，完成时刻晚于承诺的部分记为延误。目标是四台延误之和最小。

工时和承诺时刻是合成教学数据。本例用「工件放到第几个位置」的表示，不使用析取（DJ）大 M 公式。

练习用，不计正式成绩。

## 1. 模型要点

- $x_{jk}$ 为 0-1，表示样机 $j$ 是否排在第 $k$ 个位置。每台样机进入一个位置，每个位置放一台样机。
- $C_k$ 是第 $k$ 个位置做完时的时刻。$C_k = C_{k-1} + \sum_j p_j x_{jk}$，第一台从 0 时刻开始，没有释放时间。
- 延误 $T_k \ge 0$，并且 $T_k \ge C_k - \sum_j d_j x_{jk}$。
- 最小化 $\sum_k T_k$。

检测台一次只能做一台，所以位置模型自然保证了不重叠。它不描述多台机器，也不描述准备时间。

In [ ]:
from itertools import permutations
from pyscipopt import Model, quicksum

jobs = ["样机甲", "样机乙", "样机丙", "样机丁"]
process = [5, 4, 6, 3]
due = [9, 7, 15, 8]
assert all(p > 0 for p in process)

## 2. 求解位置模型

In [ ]:
def solve_single_machine(process, due):
    n = len(process)
    model = Model("IE-OPT-11 single machine tardiness")
    model.hideOutput(True)
    x = {(j, k): model.addVar(vtype="B", name=f"pos_{j}_{k}")
         for j in range(n) for k in range(n)}
    completion = {k: model.addVar(lb=0, vtype="C", name=f"C_{k}") for k in range(n)}
    tardiness = {k: model.addVar(lb=0, vtype="C", name=f"T_{k}") for k in range(n)}
    for j in range(n):
        model.addCons(quicksum(x[j, k] for k in range(n)) == 1)
    for k in range(n):
        model.addCons(quicksum(x[j, k] for j in range(n)) == 1)
        duration = quicksum(process[j] * x[j, k] for j in range(n))
        if k == 0:
            model.addCons(completion[k] == duration)
        else:
            model.addCons(completion[k] == completion[k - 1] + duration)
        model.addCons(tardiness[k] >= completion[k] - quicksum(due[j] * x[j, k] for j in range(n)))
    model.setObjective(quicksum(tardiness[k] for k in range(n)), "minimize")
    model.optimize()
    status = str(model.getStatus())
    if status != "optimal":
        raise RuntimeError(f"Solver status is {status}; optimality not certified")
    order = [next(j for j in range(n) if model.getVal(x[j, k]) > 0.5) for k in range(n)]
    finish = [float(model.getVal(completion[k])) for k in range(n)]
    late = [float(model.getVal(tardiness[k])) for k in range(n)]
    return {"objective": float(model.getObjVal()), "order": order, "finish": finish, "late": late}

result = solve_single_machine(process, due)
print("最小总延误:", result["objective"])
for k, job in enumerate(result["order"]):
    print(f"位置 {k + 1}: {jobs[job]} 完工 {result['finish'][k]:.0f} 延误 {result['late'][k]:.0f}")

## 3. 枚举四台样机的先后

$4! = 24$。对每种顺序累加工时，延误取完成时刻超出承诺的部分，再求和。这个计算不调用 SCIP。

In [ ]:
def oracle_total_tardiness(process, due):
    best = float("inf")
    best_order = None
    for order in permutations(range(len(process))):
        clock = 0
        total = 0
        for job in order:
            clock += process[job]
            total += max(0, clock - due[job])
        if total < best:
            best, best_order = total, order
    return best, best_order

oracle, oracle_order = oracle_total_tardiness(process, due)
print("独立枚举最小总延误:", oracle)
print("一条最优顺序:", oracle_order)
assert abs(result["objective"] - oracle) < 1e-6
clock = 0
recomputed = 0
for k, job in enumerate(result["order"]):
    clock += process[job]
    assert abs(result["finish"][k] - clock) < 1e-6
    recomputed += max(0, clock - due[job])
assert abs(recomputed - result["objective"]) < 1e-6
assert sorted(result["order"]) == list(range(len(jobs)))
print("通过：求解器目标与 24 种顺序一致，完工时刻与顺序匹配")

## 4. 可以接着改什么

1. 把样机丙的承诺时刻提前，看它会不会被调到更靠前的位置。
2. 若检测台有两台，位置模型要变成「分配到机器 + 机器上的位置」，即拓展为并行机调度问题，可结合分配变量与加工顺位变量联合建模。
3. 排列流水车间 Notebook 处理的是多台机器、同一工件顺序；不要把那里的最大完工时间直接当成这台检测台的总延误。